In [69]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from cmdstanpy import CmdStanModel

from lib.models.utils import (
    PerceptualModule,
    ResponseModule,
    get_stan_model_paths
)
from lib import filepaths

In [70]:
stan_model_paths = get_stan_model_paths()

dataset = pd.read_csv(filepaths.ROOT_BEHAV_DATA)

perceptual_map = dict(
    basic=PerceptualModule.BASIC,
    scpw=PerceptualModule.SCPW
)

response_map = dict(
    basic=ResponseModule.BASIC,
    brt=ResponseModule.BRT,
    brrt=ResponseModule.BRRT
)

In [71]:
perceptual_model = "scpw"
response_model = "brt"
model_type = "2level"
hierarchy = "single_subject"
n_chains = 3
n_samples = 1000
n_warmup = 200

output_dir = Path(filepaths.ROOT_STAN_MODEL_FITS)/hierarchy/model_type
os.makedirs(output_dir,exist_ok=True)

In [72]:
d = stan_model_paths[
    (stan_model_paths.perc == perceptual_map[perceptual_model]) 
    & 
    (stan_model_paths.resp == response_map[response_model])
]

stan_model_path = Path(d[f"{hierarchy}_root"].iloc[0])/model_type/d.filename.iloc[0]

model = CmdStanModel(stan_file=stan_model_path)

In [73]:
dataset = dataset[dataset.subj_id_code.isin(range(5))].reset_index(drop=True)

In [74]:
def run_single_subject_fit():
    for subj_id_code in dataset.subj_id_code.unique():
        subj_id = dataset[dataset.subj_id_code==subj_id_code].subj_id.unique()[0]
        
        df_subj = dataset[dataset.subj_id==subj_id].reset_index()
        start_idx, end_idx = np.array([
            (idx.min() + 1, idx.max() + 1) 
            for _, idx in df_subj.groupby('session_id_code').groups.items()
        ]).T

        data = dict(
            N_obs=len(df_subj.stimulus_side),
            N_sess=df_subj.session_id_code.nunique(),
            stimulus_side=df_subj.stimulus_side.values,
            stimulus_contrast=df_subj.stimulus_contrast.values,
            feedback=df_subj.feedback.values,
            choice=df_subj.choice.values,
            start_idx=start_idx,
            end_idx=end_idx
        )
        
        fit = model.sample(
            data=data, 
            chains=n_chains, 
            parallel_chains=n_chains, 
            iter_sampling=n_samples, 
            iter_warmup=n_warmup
        )
        
        np.save(output_dir/subj_id, fit)